# Chapter 4 supervised architecture comparison

**Status:** canonical thesis experiment  
**Thesis sections:** 2.5, 3, 4

Supervised learning asks each architecture to reproduce the optimized cellwise SUPG parameter. This notebook makes the common task and model interface visible; the configuration runner removes accidental differences between five copied training scripts.

## 1. The shared prediction task

For each graph, node features $x_K$ predict $\tau_K^*$, the direct-adjoint optimum produced during dataset generation. MLP has no message passing; GCN, GraphSAGE, GAT, and GATv2 use the same node targets and differ only in how information is aggregated across cell adjacency.

In [ ]:
from supgml.models import BoundedOutput, create_model

# `sample` is one saved graph. All architectures receive the same input width.
# base = create_model("gatv2", in_channels=sample.x.shape[1], hidden_channels=5, num_layers=10)
# model = BoundedOutput(base, upper="upper", method="sigmoid")
# prediction = model(sample)  # one non-negative value per mesh cell


The supervised objective is a cellwise regression loss against `graph.y`. The output constraint matters: it enforces the admissible interval established by the standard SUPG parameter, rather than relying on unconstrained predictions to remain physical.

In [ ]:
from supgml.experiments import load_config, project_root

config = load_config(project_root() / "experiments/ch4_supervised.json")
config


## 2. Keep the unsuccessful search visible

The first 64-32-32 architecture, adapted from ConvStabNet, did not produce useful solutions for any model. A narrow, deeper design—eight hidden layers of width five in the thesis account—worked better. This negative result is scientifically important: the final architecture was not an arbitrary default, and larger layers did not compensate for the small heterogeneous dataset.

## 3. What the supervised experiment showed

In the small, heterogeneous Chapter 4 setup, direct parameter regression did not consistently produce satisfactory FEM solutions. Similar target errors could have very different physical consequences, and loss scales varied substantially between problems. This failure motivates both the FEM-backed comparison and Chapter 5’s more robust common reference objective.

Run `supgml-train experiments/ch4_supervised.json` from the repository root. The runner fixes split, seed, optimizer, and output directory across MLP/GCN/GraphSAGE/GAT/GATv2; inspect its recorded history rather than comparing notebook-cell output. The long submitted traces remain in `archive/chapter4/Train_*.ipynb` as evidence, not tutorial content.